# מפגש 9 — Data Preparation חלק ב׳
יחידה 3 | מחברת מודרכת | העוגן: Data Preparation2

**בעיה:** כיצד נבחר ונתקן מאפיינים לפני חיזוי מחירי יהלומים? נלמד קורלציות, מיפוי קטגוריות, One-Hot וזיהוי מדידות חשודות. נתוני Diamonds מקורם בקישור המצגת.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df=pd.read_csv("https://raw.githubusercontent.com/arielb30/datasets/main/diamonds.csv")
print(df.shape)
display(df.head())

## 1. קורלציה: קשר אינו בהכרח סיבתיות
נחשב קשרים בין העמודות המספריות ונציג Heatmap. המשתנה price אינו בהכרח מושפע רק מ־carat.

In [ ]:
numeric=df.select_dtypes(include="number")
print(numeric.corr()["price"].sort_values(ascending=False))
plt.figure(figsize=(7,5))
sns.heatmap(numeric.corr(),annot=True,cmap="coolwarm",fmt=".2f")
plt.title("Diamonds: numeric correlations")
plt.show()

## 2. קטגוריה סדורה
`cut` מתאר איכות ליטוש בעלת סדר מקצועי. נשתמש במפה ידועה מראש, ולא במספור שרירותי.

In [ ]:
cut_order={"Fair":0,"Good":1,"Very Good":2,"Premium":3,"Ideal":4}
df["cut_rank"]=df["cut"].map(cut_order)
display(df[["cut","cut_rank"]].head())

## 3. מזהים מידות אפס ומחליטים
במצגת מוצעת מחיקת יהלומים עם x/y/z שווים לאפס. קודם נבדוק כמה כאלה קיימים; ניצור עותק מנוקה ללא שינוי של df המקורי.

In [ ]:
invalid=(df[["x","y","z"]]==0).any(axis=1)
print("Rows with zero dimension:",invalid.sum())
clean=df.loc[~invalid].copy()
print("Before:",len(df),"After:",len(clean))

## 4. כפילויות אינן בהכרח שגיאות
בדקו מספר שורות זהות. ייתכן ששני יהלומים שונים חולקים את כל המדידות, ולכן אין למחוק אוטומטית כל כפילות לפני בחינת המשמעות.

In [ ]:
print("Rows with identical values:",clean.duplicated().sum())
# רק לאחר בדיקה מקצועית של משמעות הרשומות מחליטים אם למחוק כפילויות.

## 5. One-Hot על משתנה ללא סדר — דוגמת Titanic
`Embarked` מציין נמל יציאה ללא סדר מספרי טבעי. נייצר עמודה לכל קטגוריה.

In [ ]:
titanic=pd.read_csv("https://raw.githubusercontent.com/arielb30/datasets/main/titanic_data.csv")
display(titanic[["Embarked"]].head())
encoded=pd.get_dummies(titanic[["Embarked"]],columns=["Embarked"],dtype=int)
display(encoded.head())

## 6. Data Leakage — דיון
אם יוצרים `target` מקבוצות המחיר, אסור להזין את `price` ל־X: המודל יקבל את התשובה דרך מאפיין נגזר ישירות. סכמו במשפט למה זו בעיה.

In [ ]:
small=clean[["price","carat","cut_rank"]].copy()
small["target"]=pd.qcut(small["price"],q=4,labels=[0,1,2,3])
X=small[["carat","cut_rank"]]
y=small["target"]
print(X.shape,y.shape)

## כרטיס יציאה
1. מה ההבדל בין קשר מתמטי לסיבתיות?
2. מתי map מתאים ומתי One-Hot?
3. כיצד תתעדו ערכים חריגים וכפילויות בפרויקט?
4. מדוע אסור להשתמש ב־price לחיזוי target שנגזר מהמחיר?